# Кредитный конвейер: от первого запроса до панели в BI

**Управленческий вопрос.** Где конвейер теряет время и заявки — и что исправлять в первую очередь?

Работаем только запросами: сначала смотрим, что в базе, потом проверяем данные,
потом считаем, и в конце получаем запрос, который переносится в систему дашбордов
новой панелью.

Данные — четыре выгрузки, как они приходят из систем:

| Таблица | Что в ней |
|---|---|
| `raw_applications` | заявки: продукт, канал, регион, сумма, время подачи |
| `raw_stage_events` | прохождение этапов: вход и выход по каждому этапу |
| `raw_decisions` | решения по заявкам с причиной |
| `raw_disbursements` | выдачи средств |

In [1]:
import sys
sys.path.append("..")
from tools.sqlcell import setup

setup()

База «bank_training» на связи: PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu


объект,тип
v_applications_clean,VIEW
v_initiative_cashflow,VIEW
v_initiative_fact_vs_plan,VIEW
v_initiative_npv,VIEW
v_initiative_plan,VIEW
v_pipeline_stage_sla,VIEW
v_portfolio_npv,VIEW
v_stage_events_clean,VIEW
initiative_fact,BASE TABLE
initiative_passport,BASE TABLE


## Шаг 1. Посмотреть, что вообще есть

Первый запрос к незнакомым данным — всегда про объём и границы периода.
Без этого любое число ниже повисает в воздухе.

In [2]:
%%sql
SELECT
    count(*)                        AS "Строк в выгрузке",
    count(DISTINCT application_id)  AS "Уникальных заявок",
    min(submitted_at)::date         AS "Первая заявка",
    max(submitted_at)::date         AS "Последняя заявка"
FROM raw_applications

Строк в выгрузке,Уникальных заявок,Первая заявка,Последняя заявка
3 672,3 600,2025-01-06,2025-08-03


Дальше — простейшая группировка: сколько заявок по каналам и продуктам.

In [3]:
%%sql
SELECT
    channel                                  AS "Канал",
    count(*)                                 AS "Заявок",
    round(avg(amount_requested))             AS "Средняя сумма, ₽"
FROM raw_applications
GROUP BY channel
ORDER BY "Заявок" DESC

Канал,Заявок,"Средняя сумма, ₽"
Мобильное приложение,1 230,1 511 138
Отделение,1 081,1 547 549
Партнёрская сеть,786,1 530 852
Корпоративный менеджер,575,1 497 391


## Шаг 2. Проверить данные до того, как считать

Выгрузка почти никогда не приходит чистой. Четыре проверки, которые стоит делать
всегда: повторы строк, служебные записи, невозможные значения, разрывы между
источниками.

### Повторы строк

In [4]:
%%sql
SELECT
    count(*)                                          AS "Строк",
    count(DISTINCT application_id)                    AS "Заявок",
    count(*) - count(DISTINCT application_id)         AS "Лишних строк"
FROM raw_applications

Строк,Заявок,Лишних строк
3 672,3 600,72


### Служебные записи

In [5]:
%%sql
SELECT
    count(DISTINCT application_id) AS "Тестовых заявок"
FROM raw_applications
WHERE is_test OR client_id = 'CL-TEST'

Тестовых заявок
58


### Невозможные значения и незакрытые этапы

Этап не может закончиться раньше, чем начался: такие строки означают рассинхрон
времени между системами. Пустое время выхода — это другое: этап ещё идёт на момент
выгрузки. Первое чиним, второе учитываем отдельно.

In [6]:
%%sql
SELECT
    count(*) FILTER (WHERE left_at < entered_at)  AS "Выход раньше входа",
    count(*) FILTER (WHERE left_at IS NULL)       AS "Этап не закрыт",
    count(*)                                      AS "Всего событий"
FROM raw_stage_events

Выход раньше входа,Этап не закрыт,Всего событий
171,298,14 203


### Разрыв между источниками: заявки, по которым нет решения

In [7]:
%%sql
SELECT count(*) AS "Заявок без решения"
FROM (SELECT DISTINCT application_id FROM raw_applications) a
LEFT JOIN raw_decisions d USING (application_id)
WHERE d.application_id IS NULL

Заявок без решения
54


### Сколько стоит пропустить проверку

Считаем конверсию в выдачу двумя способами: как есть и после отсева повторов и
служебных записей. Разница — цена одной пропущенной проверки.

In [8]:
%%sql
SELECT
    round(100.0 * (SELECT count(*) FROM raw_disbursements)
                / (SELECT count(*) FROM raw_applications), 1)          AS "Как есть, %",
    round(100.0 * (SELECT count(DISTINCT d.application_id)
                     FROM raw_disbursements d
                     JOIN (SELECT DISTINCT application_id
                             FROM raw_applications
                            WHERE NOT is_test AND client_id <> 'CL-TEST') a USING (application_id))
                / (SELECT count(DISTINCT application_id)
                     FROM raw_applications
                    WHERE NOT is_test AND client_id <> 'CL-TEST'), 1)  AS "После проверки, %"

"Как есть, %","После проверки, %"
"49,6","51,4"


Разница почти в два процентных пункта — на таком масштабе это сотни заявок.
А минимальная длительность этапа в исходных данных отрицательная, то есть число
не просто смещено, она невозможна.

## Шаг 3. Собрать чистый слой

Чистку не повторяют в каждом запросе — её один раз оформляют представлением.
Дальше все расчёты идут поверх него, и правило чистки живёт в одном месте.

In [9]:
%%sql
CREATE OR REPLACE VIEW v_applications_clean AS
SELECT DISTINCT ON (application_id)
    application_id,
    client_id,
    product,
    channel,
    region,
    amount_requested,
    submitted_at,
    source_system
FROM raw_applications
WHERE NOT is_test
  AND client_id <> 'CL-TEST'
ORDER BY application_id, submitted_at

CREATE VIEW


In [10]:
%%sql
CREATE OR REPLACE VIEW v_stage_events_clean AS
SELECT
    e.event_id,
    e.application_id,
    e.stage,
    e.entered_at,
    e.left_at,
    EXTRACT(EPOCH FROM (e.left_at - e.entered_at)) / 3600 AS hours
FROM raw_stage_events e
JOIN v_applications_clean a USING (application_id)
WHERE e.left_at IS NOT NULL
  AND e.left_at > e.entered_at

CREATE VIEW


Проверяем, что чистый слой не потерял лишнего.

In [11]:
%%sql
SELECT
    (SELECT count(*) FROM raw_applications)        AS "Строк было",
    (SELECT count(*) FROM v_applications_clean)    AS "Заявок стало",
    (SELECT count(*) FROM raw_stage_events)        AS "Событий было",
    (SELECT count(*) FROM v_stage_events_clean)    AS "Событий стало"

Строк было,Заявок стало,Событий было,Событий стало
3 672,3 542,14 203,13 516


## Шаг 4. Посчитать то, ради чего всё затевалось

### Куда доходят заявки

In [12]:
%%sql
SELECT
    e.stage                                                             AS "Этап",
    count(DISTINCT e.application_id)                                    AS "Дошло заявок",
    round(100.0 * count(DISTINCT e.application_id)
          / (SELECT count(*) FROM v_applications_clean), 1)             AS "Доля от поданных, %"
FROM v_stage_events_clean e
GROUP BY e.stage
ORDER BY "Дошло заявок" DESC

Этап,Дошло заявок,"Доля от поданных, %"
Приём заявки,3 417,"96,5"
Скоринг,3 306,"93,3"
Андеррайтинг,2 654,"74,9"
Подготовка документов,2 143,"60,5"
Выдача,1 996,"56,4"


### Сколько времени занимает этап

Среднее по длительностям обманывает: хвост тянет его вверх. Смотрим медиану и
девяностую перцентиль — типичный срок и срок, в который укладывается почти всё.

In [13]:
%%sql
SELECT
    stage                                                                        AS "Этап",
    count(*)                                                                     AS "Событий",
    round(percentile_cont(0.5) WITHIN GROUP (ORDER BY hours)::numeric, 1)        AS "Медиана, ч",
    round(percentile_cont(0.9) WITHIN GROUP (ORDER BY hours)::numeric, 1)        AS "P90, ч"
FROM v_stage_events_clean
GROUP BY stage
ORDER BY "Медиана, ч" DESC

Этап,Событий,"Медиана, ч","P90, ч"
Андеррайтинг,2 654,"40,6",78
Подготовка документов,2 143,13,"23,7"
Выдача,1 996,"8,1",14
Скоринг,3 306,"2,8","5,1"
Приём заявки,3 417,2,"3,8"


Самый долгий этап виден сразу. Следующий вопрос — одинаково ли он долгий везде.

In [14]:
%%sql
SELECT
    a.channel                                                                    AS "Канал",
    count(*)                                                                     AS "Заявок",
    round(percentile_cont(0.5) WITHIN GROUP (ORDER BY e.hours)::numeric, 1)      AS "Медиана, ч",
    round(percentile_cont(0.9) WITHIN GROUP (ORDER BY e.hours)::numeric, 1)      AS "P90, ч"
FROM v_stage_events_clean e
JOIN v_applications_clean a USING (application_id)
WHERE e.stage = 'Андеррайтинг'
GROUP BY a.channel
ORDER BY "Медиана, ч" DESC

Канал,Заявок,"Медиана, ч","P90, ч"
Партнёрская сеть,571,"54,9","106,9"
Мобильное приложение,904,"37,9","69,1"
Отделение,764,"37,2","66,5"
Корпоративный менеджер,415,"36,7","62,7"


### Где теряются заявки

Причины отказов показывают, что именно исправлять: скоринговый порог, документы
или работу с клиентом.

In [15]:
%%sql
SELECT
    d.reason                                                        AS "Причина",
    count(*)                                                        AS "Заявок",
    round(100.0 * count(*) / sum(count(*)) OVER (), 1)              AS "Доля, %"
FROM raw_decisions d
JOIN v_applications_clean a USING (application_id)
WHERE d.decision = 'Отказ'
GROUP BY d.reason
ORDER BY "Заявок" DESC

Причина,Заявок,"Доля, %"
Скоринговый балл ниже порога,679,"40,2"
Высокая долговая нагрузка,495,"29,3"
Клиент не явился за средствами,234,"13,9"
Клиент отозвал заявку,163,"9,7"
Неполный пакет документов,118,7


## Шаг 5. Запрос, который уходит в панель

Разовый ответ в тетради живёт до закрытия ноутбука. Чтобы им пользовались,
запрос переносят в систему дашбордов. Ниже — тот самый запрос: срок каждого
этапа в разрезе канала.

In [16]:
%%sql
SELECT
    a.channel                                                                    AS "Канал",
    e.stage                                                                      AS "Этап",
    count(*)                                                                     AS "Заявок",
    round(percentile_cont(0.5) WITHIN GROUP (ORDER BY e.hours)::numeric, 1)      AS "Медиана, ч",
    round(percentile_cont(0.9) WITHIN GROUP (ORDER BY e.hours)::numeric, 1)      AS "P90, ч"
FROM v_stage_events_clean e
JOIN v_applications_clean a USING (application_id)
GROUP BY a.channel, e.stage
ORDER BY "Медиана, ч" DESC

Канал,Этап,Заявок,"Медиана, ч","P90, ч"
Партнёрская сеть,Андеррайтинг,571,"54,9","106,9"
Мобильное приложение,Андеррайтинг,904,"37,9","69,1"
Отделение,Андеррайтинг,764,"37,2","66,5"
Корпоративный менеджер,Андеррайтинг,415,"36,7","62,7"
Мобильное приложение,Подготовка документов,727,"13,2","22,9"
Корпоративный менеджер,Подготовка документов,334,13,"24,1"
Отделение,Подготовка документов,618,"12,9","23,6"
Партнёрская сеть,Подготовка документов,464,"12,7","24,7"
Партнёрская сеть,Выдача,429,"8,5",14
Мобильное приложение,Выдача,696,"8,1","14,2"


Тот же расчёт оформляем представлением — тогда панель ссылается на него, а правило
расчёта остаётся в базе и не расходится между отчётами.

In [17]:
%%sql
CREATE OR REPLACE VIEW v_pipeline_stage_sla AS
SELECT
    a.channel                                                              AS channel,
    e.stage                                                                AS stage,
    count(*)                                                               AS applications,
    round(percentile_cont(0.5) WITHIN GROUP (ORDER BY e.hours)::numeric, 1) AS median_hours,
    round(percentile_cont(0.9) WITHIN GROUP (ORDER BY e.hours)::numeric, 1) AS p90_hours
FROM v_stage_events_clean e
JOIN v_applications_clean a USING (application_id)
GROUP BY a.channel, e.stage

CREATE VIEW


In [18]:
%%sql
SELECT * FROM v_pipeline_stage_sla ORDER BY median_hours DESC

channel,stage,applications,median_hours,p90_hours
Партнёрская сеть,Андеррайтинг,571,"54,9","106,9"
Мобильное приложение,Андеррайтинг,904,"37,9","69,1"
Отделение,Андеррайтинг,764,"37,2","66,5"
Корпоративный менеджер,Андеррайтинг,415,"36,7","62,7"
Мобильное приложение,Подготовка документов,727,"13,2","22,9"
Корпоративный менеджер,Подготовка документов,334,13,"24,1"
Отделение,Подготовка документов,618,"12,9","23,6"
Партнёрская сеть,Подготовка документов,464,"12,7","24,7"
Партнёрская сеть,Выдача,429,"8,5",14
Мобильное приложение,Выдача,696,"8,1","14,2"


## Перенести запрос в панель

1. Откройте систему дашбордов: <http://localhost:3000>
2. **Создать → Запрос → Свой SQL-запрос**, источник «Учебная база банка».
3. Вставьте запрос из ячейки выше и выполните.
4. Смените тип отображения на столбчатую диаграмму: по горизонтали — этап,
   по вертикали — медиана, разбивка по каналу.
5. Сохраните под именем «Сроки этапов конвейера» и добавьте на новый дашборд
   «Кредитный конвейер».

Если данные в базе обновятся, панель покажет новые значения без единой правки —
именно поэтому расчёт переносят в BI, а не рассылают картинку.

## Что из этого следует

- Самый долгий этап конвейера — андеррайтинг, и он неодинаков по каналам:
  партнёрская сеть ждёт заметно дольше остальных.
- Проверка данных меняет управленческий вывод: конверсия «как есть» и конверсия
  после отсева повторов отличаются почти на два процентных пункта.
- Правило чистки живёт в представлении, а не в переписке: любой следующий запрос
  считает по тем же правилам.

**Задание.** Возьмите свой процесс и повторите путь: обзор → проверки → чистый
слой → медиана и P90 по этапам → запрос в панель. Достаточно одного показателя,
который вы готовы обсуждать с командой.